# 🏛️ Indonesian Anti-Corruption Law NLP
### Analisis Normatif Hukum Korupsi Indonesia vs Standar UNCAC

**Pipeline:**
```
PDF UU → Parsing → JSON Terstruktur → Embedding → Topic Modeling → Gap Analysis
```

**Dokumen yang digunakan:**
- `UN_Convention_Against_Corruption.pdf` (UNCAC - Standar Internasional)
- `UU Nomor 7 Tahun 2006` (Ratifikasi UNCAC)
- `UU Nomor 31 Tahun 1999` (UU Tipikor)
- `UU Nomor 20 Tahun 2001` (Amandemen UU Tipikor)
- `UU Nomor 28 Tahun 1999` (Penyelenggaraan Negara Bersih KKN)
- `UU Nomor 30 Tahun 2002` (Pembentukan KPK)
- `UU Nomor 19 Tahun 2019` (Revisi UU KPK)

---

## CELL 1 — Instalasi Library

In [2]:
# ============================================================
# CELL 1 — INSTALASI LIBRARY
# Jalankan sekali saja, kemudian restart runtime jika diminta
# ============================================================

print('📦 Menginstall library yang dibutuhkan...')

!pip install -q pdfplumber
!pip install -q sentence-transformers
!pip install -q bertopic
!pip install -q keybert
!pip install -q plotly
!pip install -q umap-learn
!pip install -q hdbscan

print('✅ Semua library berhasil diinstall!')

📦 Menginstall library yang dibutuhkan...
✅ Semua library berhasil diinstall!


## CELL 2 — Upload & Konfigurasi Dokumen

In [3]:
# ============================================================
# CELL 2 — UPLOAD DOKUMEN PDF
# Upload semua 7 file PDF dari komputer Anda
# ============================================================

from google.colab import files
import os

print('📂 Silakan upload semua 7 file PDF...')
print('   - UN_Convention_Against_Corruption.pdf')
print('   - UU Nomor 7 Tahun 2006.pdf')
print('   - UU Nomor 31 Tahun 1999.pdf')
print('   - UU Nomor 20 Tahun 2001.pdf')
print('   - UU Nomor 28 Tahun 1999.pdf')
print('   - UU Nomor 30 Tahun 2002.pdf')
print('   - UU Nomor 19 Tahun 2019.pdf')
print()

uploaded = files.upload()

# Tampilkan file yang berhasil diupload
print(f'\n✅ {len(uploaded)} file berhasil diupload:')
for fname in uploaded.keys():
    size_kb = len(uploaded[fname]) / 1024
    print(f'   📄 {fname} ({size_kb:.1f} KB)')

# Konfigurasi metadata dokumen
DOC_CONFIG = {
    'UN_Convention_Against_Corruption.pdf': {
        'label'   : 'UNCAC',
        'bahasa'  : 'en',
        'jenis'   : 'internasional',
        'tahun'   : 2003
    },
    'UU Nomor 7 Tahun 2006.pdf': {
        'label'   : 'UU_7_2006',
        'bahasa'  : 'id',
        'jenis'   : 'ratifikasi',
        'tahun'   : 2006
    },
    'UU Nomor 31 Tahun 1999.pdf': {
        'label'   : 'UU_31_1999',
        'bahasa'  : 'id',
        'jenis'   : 'tipikor',
        'tahun'   : 1999
    },
    'UU Nomor 20 Tahun 2001.pdf': {
        'label'   : 'UU_20_2001',
        'bahasa'  : 'id',
        'jenis'   : 'tipikor_amandemen',
        'tahun'   : 2001
    },
    'UU Nomor 28 Tahun 1999.pdf': {
        'label'   : 'UU_28_1999',
        'bahasa'  : 'id',
        'jenis'   : 'kkn',
        'tahun'   : 1999
    },
    'UU Nomor 30 Tahun 2002.pdf': {
        'label'   : 'UU_30_2002',
        'bahasa'  : 'id',
        'jenis'   : 'kpk',
        'tahun'   : 2002
    },
    'UU Nomor 19 Tahun 2019.pdf': {
        'label'   : 'UU_19_2019',
        'bahasa'  : 'id',
        'jenis'   : 'kpk_revisi',
        'tahun'   : 2019
    }
}

print('\n📋 Konfigurasi dokumen siap.')

📂 Silakan upload semua 7 file PDF...
   - UN_Convention_Against_Corruption.pdf
   - UU Nomor 7 Tahun 2006.pdf
   - UU Nomor 31 Tahun 1999.pdf
   - UU Nomor 20 Tahun 2001.pdf
   - UU Nomor 28 Tahun 1999.pdf
   - UU Nomor 30 Tahun 2002.pdf
   - UU Nomor 19 Tahun 2019.pdf



Saving UN_Convention_Against_Corruption.pdf to UN_Convention_Against_Corruption.pdf
Saving UU Nomor 7 Tahun 2006.pdf to UU Nomor 7 Tahun 2006.pdf
Saving UU Nomor 19 Tahun 2019.pdf to UU Nomor 19 Tahun 2019.pdf
Saving UU Nomor 20 Tahun 2001.pdf to UU Nomor 20 Tahun 2001.pdf
Saving UU Nomor 28 Tahun 1999.pdf to UU Nomor 28 Tahun 1999.pdf
Saving UU Nomor 30 Tahun 2002.pdf to UU Nomor 30 Tahun 2002.pdf
Saving UU Nomor 31 Tahun 1999.pdf to UU Nomor 31 Tahun 1999.pdf

✅ 7 file berhasil diupload:
   📄 UN_Convention_Against_Corruption.pdf (429.6 KB)
   📄 UU Nomor 7 Tahun 2006.pdf (20.6 KB)
   📄 UU Nomor 19 Tahun 2019.pdf (231.9 KB)
   📄 UU Nomor 20 Tahun 2001.pdf (109.1 KB)
   📄 UU Nomor 28 Tahun 1999.pdf (90.2 KB)
   📄 UU Nomor 30 Tahun 2002.pdf (111.7 KB)
   📄 UU Nomor 31 Tahun 1999.pdf (111.4 KB)

📋 Konfigurasi dokumen siap.


##  CELL 3 — Preprocessing: Ekstraksi Teks dari PDF

In [4]:
# ============================================================
# CELL 3 — EKSTRAKSI TEKS DARI PDF
# Menggunakan pdfplumber untuk membaca setiap halaman
# ============================================================

import pdfplumber
import re
import json

def ekstrak_teks_pdf(filepath):
    """Ekstrak seluruh teks dari PDF dengan pdfplumber."""
    teks_semua = []
    try:
        with pdfplumber.open(filepath) as pdf:
            for i, halaman in enumerate(pdf.pages):
                teks = halaman.extract_text()
                if teks:
                    teks_semua.append(teks.strip())
    except Exception as e:
        print(f'   ⚠️  Error pada {filepath}: {e}')
    return '\n'.join(teks_semua)


def bersihkan_teks(teks):
    """Bersihkan teks dari karakter noise hasil OCR/PDF."""
    # Hapus baris kosong berlebih
    teks = re.sub(r'\n{3,}', '\n\n', teks)
    # Hapus spasi berlebih
    teks = re.sub(r'[ \t]+', ' ', teks)
    # Hapus karakter non-printable
    teks = re.sub(r'[^\x20-\x7E\n\t\u00C0-\u024F\u0100-\u024F]', '', teks)
    return teks.strip()


# Proses semua dokumen yang diupload
print('🔍 Memulai ekstraksi teks dari semua PDF...\n')

corpus_raw = {}

for filename, config in DOC_CONFIG.items():
    if filename in uploaded:
        print(f'   📄 Memproses: {filename}')
        teks = ekstrak_teks_pdf(filename)
        teks = bersihkan_teks(teks)
        corpus_raw[config['label']] = {
            'filename' : filename,
            'config'   : config,
            'teks'     : teks,
            'panjang'  : len(teks),
            'kata'     : len(teks.split())
        }
        print(f'      ✅ {len(teks.split()):,} kata diekstrak')
    else:
        print(f'   ❌ {filename} tidak ditemukan dalam upload')

print(f'\n📊 Total dokumen berhasil diproses: {len(corpus_raw)}')
total_kata = sum(d["kata"] for d in corpus_raw.values())
print(f'📊 Total kata dalam corpus: {total_kata:,}')

🔍 Memulai ekstraksi teks dari semua PDF...

   📄 Memproses: UN_Convention_Against_Corruption.pdf
      ✅ 20,975 kata diekstrak
   📄 Memproses: UU Nomor 7 Tahun 2006.pdf
      ✅ 1,474 kata diekstrak
   📄 Memproses: UU Nomor 31 Tahun 1999.pdf
      ✅ 5,612 kata diekstrak
   📄 Memproses: UU Nomor 20 Tahun 2001.pdf
      ✅ 3,604 kata diekstrak
   📄 Memproses: UU Nomor 28 Tahun 1999.pdf
      ✅ 3,901 kata diekstrak
   📄 Memproses: UU Nomor 30 Tahun 2002.pdf
      ✅ 8,778 kata diekstrak
   📄 Memproses: UU Nomor 19 Tahun 2019.pdf
      ✅ 5,123 kata diekstrak

📊 Total dokumen berhasil diproses: 7
📊 Total kata dalam corpus: 49,467


## CELL 4 — Parsing: Ekstraksi Per Pasal

In [5]:
# ============================================================
# CELL 4 — PARSING PER PASAL
# Memisahkan teks menjadi unit pasal menggunakan regex
# ============================================================

def parse_pasal_indonesia(teks, label_dokumen):
    """
    Parsing teks UU Indonesia menjadi list pasal.
    Pola yang dikenali:
    - 'Pasal 1', 'Pasal 12', dst
    """
    pasal_list = []

    # Pisahkan berdasarkan penanda pasal
    pattern = r'(Pasal\s+\d+[A-Z]?)'
    bagian = re.split(pattern, teks)

    i = 1
    while i < len(bagian) - 1:
        header = bagian[i].strip()     # "Pasal 5"
        isi    = bagian[i+1].strip()   # isi pasal

        # Ekstrak nomor pasal
        nomor_match = re.search(r'(\d+[A-Z]?)', header)
        nomor = nomor_match.group(1) if nomor_match else '?'

        if len(isi) > 20:  # filter pasal kosong
            pasal_list.append({
                'id'        : f'{label_dokumen}_pasal_{nomor}',
                'dokumen'   : label_dokumen,
                'pasal'     : nomor,
                'teks_penuh': f'{header}\n{isi}',
                'isi'       : isi[:1000],   # batasi 1000 karakter untuk embedding
                'bahasa'    : 'id'
            })
        i += 2

    return pasal_list


def parse_article_uncac(teks):
    """
    Parsing teks UNCAC (Inggris) menjadi list article.
    Pola: 'Article 1', 'Article 12', dst
    """
    article_list = []

    pattern = r'(Article\s+\d+[A-Z]?[\.\s])'
    bagian = re.split(pattern, teks)

    i = 1
    while i < len(bagian) - 1:
        header = bagian[i].strip()
        isi    = bagian[i+1].strip()

        nomor_match = re.search(r'(\d+)', header)
        nomor = nomor_match.group(1) if nomor_match else '?'

        if len(isi) > 20:
            article_list.append({
                'id'        : f'UNCAC_article_{nomor}',
                'dokumen'   : 'UNCAC',
                'pasal'     : nomor,
                'teks_penuh': f'{header}\n{isi}',
                'isi'       : isi[:1000],
                'bahasa'    : 'en'
            })
        i += 2

    return article_list


# Jalankan parsing untuk semua dokumen
print('📐 Memulai parsing per pasal...\n')

corpus_pasal = []   # semua pasal dari semua dokumen
pasal_per_dok = {}  # pasal per dokumen

for label, data in corpus_raw.items():
    if label == 'UNCAC':
        pasal = parse_article_uncac(data['teks'])
    else:
        pasal = parse_pasal_indonesia(data['teks'], label)

    pasal_per_dok[label] = pasal
    corpus_pasal.extend(pasal)
    print(f'   📜 {label}: {len(pasal)} pasal/artikel ditemukan')

print(f'\n✅ Total unit pasal dalam corpus: {len(corpus_pasal)}')

📐 Memulai parsing per pasal...

   📜 UNCAC: 71 pasal/artikel ditemukan
   📜 UU_7_2006: 9 pasal/artikel ditemukan
   📜 UU_31_1999: 90 pasal/artikel ditemukan
   📜 UU_20_2001: 36 pasal/artikel ditemukan
   📜 UU_28_1999: 50 pasal/artikel ditemukan
   📜 UU_30_2002: 127 pasal/artikel ditemukan
   📜 UU_19_2019: 118 pasal/artikel ditemukan

✅ Total unit pasal dalam corpus: 501


## CELL 5 — Simpan Corpus Terstruktur ke JSON

In [6]:
# ============================================================
# CELL 5 — SIMPAN CORPUS KE JSON
# Output: corpus_structured.json
# ============================================================

import json
from datetime import datetime

output = {
    'metadata': {
        'project'        : 'Indonesian Anti-Corruption Law NLP',
        'dibuat_pada'    : datetime.now().isoformat(),
        'total_dokumen'  : len(corpus_raw),
        'total_pasal'    : len(corpus_pasal),
        'dokumen'        : list(corpus_raw.keys())
    },
    'ringkasan_per_dokumen': {
        label: {
            'jumlah_pasal' : len(pasal_per_dok.get(label, [])),
            'total_kata'   : data['kata'],
            'bahasa'       : data['config']['bahasa'],
            'tahun'        : data['config']['tahun']
        }
        for label, data in corpus_raw.items()
    },
    'corpus': corpus_pasal
}

with open('corpus_structured.json', 'w', encoding='utf-8') as f:
    json.dump(output, f, ensure_ascii=False, indent=2)

print('💾 corpus_structured.json berhasil disimpan!')
print()
print('📊 RINGKASAN CORPUS:')
print('=' * 50)
for label, info in output['ringkasan_per_dokumen'].items():
    print(f'  {label:<20} | {info["jumlah_pasal"]:>4} pasal | {info["total_kata"]:>7,} kata | {info["bahasa"].upper()}')
print('=' * 50)
print(f'  {"TOTAL":<20} | {len(corpus_pasal):>4} pasal | {sum(d["kata"] for d in corpus_raw.values()):>7,} kata')

# Preview 1 pasal sebagai contoh
print('\n📋 Contoh struktur pasal (pasal pertama):')
print(json.dumps(corpus_pasal[0], indent=2, ensure_ascii=False))

💾 corpus_structured.json berhasil disimpan!

📊 RINGKASAN CORPUS:
  UNCAC                |   71 pasal |  20,975 kata | EN
  UU_7_2006            |    9 pasal |   1,474 kata | ID
  UU_31_1999           |   90 pasal |   5,612 kata | ID
  UU_20_2001           |   36 pasal |   3,604 kata | ID
  UU_28_1999           |   50 pasal |   3,901 kata | ID
  UU_30_2002           |  127 pasal |   8,778 kata | ID
  UU_19_2019           |  118 pasal |   5,123 kata | ID
  TOTAL                |  501 pasal |  49,467 kata

📋 Contoh struktur pasal (pasal pertama):
{
  "id": "UNCAC_article_1",
  "dokumen": "UNCAC",
  "pasal": "1",
  "teks_penuh": "Article 1.\nStatement of purpose\nThe purposes of this Convention are:\n(a) To promote and strengthen measures to prevent and combat corrup-\ntion more efficiently and effectively;\n(b) To promote, facilitate and support international cooperation and\ntechnical assistance in the prevention of and fight against corruption, including\nin asset recovery;\n(c) To prom

## CELL 6 — KeyBERT: Ekstraksi Istilah Hukum Kunci

In [7]:
# ============================================================
# CELL 6 — KEYBERT: EKSTRAKSI TERMINOLOGI HUKUM
# Zero-shot, tanpa anotasi manual
# Model: paraphrase-multilingual-MiniLM-L12-v2
# ============================================================

from keybert import KeyBERT
from collections import Counter
import pandas as pd

print('🔑 Memuat model KeyBERT (multilingual)...')
kw_model = KeyBERT(model='paraphrase-multilingual-MiniLM-L12-v2')
print('✅ Model KeyBERT siap!')


def ekstrak_istilah(teks_list, top_n=10, ngram_range=(1, 3)):
    """Ekstrak istilah kunci dari list teks."""
    teks_gabung = ' '.join(teks_list)
    keywords = kw_model.extract_keywords(
        teks_gabung,
        keyphrase_ngram_range=ngram_range,
        stop_words=None,
        top_n=top_n,
        use_mmr=True,
        diversity=0.5
    )
    return keywords


print('\n🔍 Mengekstrak istilah hukum per dokumen...\n')

hasil_keywords = {}

for label, pasal_list in pasal_per_dok.items():
    if not pasal_list:
        continue
    teks_list = [p['isi'] for p in pasal_list]
    keywords  = ekstrak_istilah(teks_list, top_n=15)
    hasil_keywords[label] = keywords

    print(f'📄 {label} — Top 10 Istilah Hukum:')
    for term, score in keywords[:10]:
        bar = '█' * int(score * 20)
        print(f'   {term:<40} {bar} {score:.3f}')
    print()

# Istilah lintas dokumen
print('\n🌐 ISTILAH HUKUM PALING UMUM (Lintas Dokumen):')
print('=' * 55)
semua_term = []
for kw_list in hasil_keywords.values():
    semua_term.extend([t for t, s in kw_list])
counter = Counter(semua_term)
for term, freq in counter.most_common(15):
    print(f'   {term:<40} muncul di {freq} dokumen')

🔑 Memuat model KeyBERT (multilingual)...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/526 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.08M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

✅ Model KeyBERT siap!

🔍 Mengekstrak istilah hukum per dokumen...

📄 UNCAC — Top 10 Istilah Hukum:
   statement of purpose                     ██████████████ 0.704
   duties the solicitation                  ███████████ 0.589
   purpose of prevent                       ██████████ 0.538
   public function including                ██████████ 0.523
   law to promote                           ██████████ 0.510
   constitutional principles an             ██████████ 0.506
   transparency and objective               █████████ 0.490
   territorial integrity of                 █████████ 0.485
   corruption to enhance                    █████████ 0.475
   codes or standards                       ████████ 0.430

📄 UU_7_2006 — Top 10 Istilah Hukum:
   undang republik indonesia                ███████████████ 0.766
   indonesia tahun 2000                     █████████████ 0.673
   2003 konvensi perserikatan               ███████████ 0.596
   perjanjian internasional lembaran        ███████████ 0.594


## CELL 7 — LaBSE: Embedding Cross-lingual EN ↔ ID

In [8]:
# ============================================================
# CELL 7 — LaBSE EMBEDDING
# Membuat vektor semantik untuk setiap pasal
# Support cross-lingual: Inggris (UNCAC) ↔ Indonesia (UU)
# ============================================================

from sentence_transformers import SentenceTransformer
import numpy as np

print('🧠 Memuat model LaBSE...')
print('   (Language-agnostic BERT Sentence Embeddings)')
print('   Mendukung 109 bahasa termasuk EN dan ID')

# LaBSE adalah model terbaik untuk cross-lingual similarity
model_embedding = SentenceTransformer('sentence-transformers/LaBSE')
print('✅ Model LaBSE siap!')

print('\n⚙️  Membuat embedding untuk semua pasal...')
print(f'   Total pasal: {len(corpus_pasal)}')

# Ambil teks isi dari semua pasal
teks_untuk_embed = [p['isi'] for p in corpus_pasal]

# Buat embedding (proses batch untuk efisiensi)
embeddings = model_embedding.encode(
    teks_untuk_embed,
    batch_size=16,
    show_progress_bar=True,
    normalize_embeddings=True   # normalisasi untuk cosine similarity
)

print(f'\n✅ Embedding selesai!')
print(f'   Shape matrix embedding: {embeddings.shape}')
print(f'   Dimensi vektor per pasal: {embeddings.shape[1]}')

# Simpan embedding ke numpy array
np.save('embeddings_labse.npy', embeddings)
print('   💾 Embedding disimpan ke embeddings_labse.npy')

🧠 Memuat model LaBSE...
   (Language-agnostic BERT Sentence Embeddings)
   Mendukung 109 bahasa termasuk EN dan ID


modules.json:   0%|          | 0.00/461 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

config.json:   0%|          | 0.00/114 [00:00<?, ?B/s]

2_Dense/model.safetensors:   0%|          | 0.00/2.36M [00:00<?, ?B/s]

✅ Model LaBSE siap!

⚙️  Membuat embedding untuk semua pasal...
   Total pasal: 501


Batches:   0%|          | 0/32 [00:00<?, ?it/s]


✅ Embedding selesai!
   Shape matrix embedding: (501, 768)
   Dimensi vektor per pasal: 768
   💾 Embedding disimpan ke embeddings_labse.npy


## CELL 8 — Gap Analysis: UNCAC vs UU Indonesia

In [9]:
# ============================================================
# CELL 8 — GAP ANALYSIS
# Membandingkan pasal UNCAC dengan UU Indonesia
# Menggunakan cosine similarity dari LaBSE embedding
# ============================================================

from sklearn.metrics.pairwise import cosine_similarity
import pandas as pd

# Pisahkan index UNCAC dan UU Indonesia
idx_uncac = [i for i, p in enumerate(corpus_pasal) if p['dokumen'] == 'UNCAC']
idx_uu_id = [i for i, p in enumerate(corpus_pasal) if p['dokumen'] != 'UNCAC']

print(f'📊 UNCAC articles: {len(idx_uncac)}')
print(f'📊 Pasal UU Indonesia: {len(idx_uu_id)}')

# Ambil embedding masing-masing
embed_uncac = embeddings[idx_uncac]
embed_uu_id = embeddings[idx_uu_id]

# Hitung cosine similarity matrix
print('\n⚙️  Menghitung similarity matrix UNCAC ↔ UU Indonesia...')
sim_matrix = cosine_similarity(embed_uncac, embed_uu_id)
print(f'   Shape similarity matrix: {sim_matrix.shape}')

# Untuk setiap artikel UNCAC, cari padanan terbaik di UU Indonesia
THRESHOLD_TINGGI  = 0.80  # sangat mirip = diadopsi
THRESHOLD_SEDANG  = 0.65  # cukup mirip = diadopsi sebagian
# di bawah 0.65 = GAP (belum diadopsi)

gap_results = []

for i, uncac_idx in enumerate(idx_uncac):
    pasal_uncac   = corpus_pasal[uncac_idx]
    skor_sim      = sim_matrix[i]
    best_match_j  = np.argmax(skor_sim)
    best_score    = skor_sim[best_match_j]
    best_uu_idx   = idx_uu_id[best_match_j]
    best_uu_pasal = corpus_pasal[best_uu_idx]

    # Tentukan status adopsi
    if best_score >= THRESHOLD_TINGGI:
        status = 'Diadopsi Penuh'
        emoji  = '✅'
    elif best_score >= THRESHOLD_SEDANG:
        status = 'Diadopsi Sebagian'
        emoji  = '🟡'
    else:
        status = 'GAP - Belum Diadopsi'
        emoji  = '❌'

    gap_results.append({
        'uncac_article'  : f"Article {pasal_uncac['pasal']}",
        'uncac_preview'  : pasal_uncac['isi'][:100] + '...',
        'padanan_terbaik': f"{best_uu_pasal['dokumen']} Pasal {best_uu_pasal['pasal']}",
        'similarity'     : round(float(best_score), 4),
        'status'         : status,
        'emoji'          : emoji
    })

# Buat DataFrame dan tampilkan
df_gap = pd.DataFrame(gap_results)
df_gap_sorted = df_gap.sort_values('similarity', ascending=True)

print('\n' + '=' * 70)
print('📋 HASIL GAP ANALYSIS: UNCAC vs UU INDONESIA')
print('=' * 70)

# Hitung ringkasan
n_penuh   = len(df_gap[df_gap['status'] == 'Diadopsi Penuh'])
n_sebagian= len(df_gap[df_gap['status'] == 'Diadopsi Sebagian'])
n_gap     = len(df_gap[df_gap['status'] == 'GAP - Belum Diadopsi'])

print(f'  ✅ Diadopsi Penuh     : {n_penuh} artikel')
print(f'  🟡 Diadopsi Sebagian  : {n_sebagian} artikel')
print(f'  ❌ GAP (belum diadopsi): {n_gap} artikel')
print('=' * 70)

# Tampilkan artikel dengan similarity terendah (GAP terbesar)
print('\n🚨 TOP 10 GAP TERBESAR (UNCAC belum diadopsi Indonesia):')
print('-' * 70)
for _, row in df_gap_sorted.head(10).iterrows():
    print(f"  {row['emoji']} {row['uncac_article']:<15} similarity: {row['similarity']:.3f}  → {row['status']}")
    print(f"     Padanan: {row['padanan_terbaik']}")
    print()

# Simpan hasil
df_gap.to_csv('gap_analysis_result.csv', index=False, encoding='utf-8')
print('💾 gap_analysis_result.csv berhasil disimpan!')

📊 UNCAC articles: 71
📊 Pasal UU Indonesia: 430

⚙️  Menghitung similarity matrix UNCAC ↔ UU Indonesia...
   Shape similarity matrix: (71, 430)

📋 HASIL GAP ANALYSIS: UNCAC vs UU INDONESIA
  ✅ Diadopsi Penuh     : 0 artikel
  🟡 Diadopsi Sebagian  : 41 artikel
  ❌ GAP (belum diadopsi): 30 artikel

🚨 TOP 10 GAP TERBESAR (UNCAC belum diadopsi Indonesia):
----------------------------------------------------------------------
  ❌ Article 51      similarity: 0.451  → GAP - Belum Diadopsi
     Padanan: UU_7_2006 Pasal 66

  ❌ Article 59      similarity: 0.479  → GAP - Belum Diadopsi
     Padanan: UU_7_2006 Pasal 66

  ❌ Article 45      similarity: 0.501  → GAP - Belum Diadopsi
     Padanan: UU_20_2001 Pasal 12

  ❌ Article 71      similarity: 0.508  → GAP - Belum Diadopsi
     Padanan: UU_7_2006 Pasal 2

  ❌ Article 70      similarity: 0.514  → GAP - Belum Diadopsi
     Padanan: UU_19_2019 Pasal 40

  ❌ Article 28      similarity: 0.538  → GAP - Belum Diadopsi
     Padanan: UU_31_1999 Pasal 2


## CELL 9 — Topic Modeling dengan BERTopic

In [10]:
# ============================================================
# CELL 9 — BERTOPIC: TOPIC MODELING
# Menemukan klaster tema hukum korupsi secara otomatis
# Tanpa anotasi manual
# ============================================================

from bertopic import BERTopic
from umap import UMAP
from hdbscan import HDBSCAN

print('🗺️  Mempersiapkan BERTopic...')

# Konfigurasi UMAP untuk reduksi dimensi
umap_model = UMAP(
    n_components=5,
    n_neighbors=10,
    min_dist=0.0,
    random_state=42
)

# Konfigurasi HDBSCAN untuk clustering
hdbscan_model = HDBSCAN(
    min_cluster_size=3,
    min_samples=2,
    prediction_data=True
)

# Inisialisasi BERTopic
topic_model = BERTopic(
    umap_model=umap_model,
    hdbscan_model=hdbscan_model,
    nr_topics='auto',
    verbose=True
)

# Gunakan teks semua pasal (ID saja untuk topic modeling)
teks_id_only = [
    p['isi'] for p in corpus_pasal
    if p['bahasa'] == 'id' and len(p['isi']) > 50
]

print(f'\n⚙️  Menjalankan topic modeling pada {len(teks_id_only)} pasal (Bahasa Indonesia)...')

topics, probs = topic_model.fit_transform(teks_id_only)

print('\n✅ Topic modeling selesai!')

# Tampilkan topik yang ditemukan
print('\n📊 TOPIK HUKUM YANG DITEMUKAN:')
print('=' * 60)
topic_info = topic_model.get_topic_info()

for _, row in topic_info.iterrows():
    if row['Topic'] == -1:
        continue  # skip outlier
    topic_id   = row['Topic']
    count      = row['Count']
    top_words  = topic_model.get_topic(topic_id)
    words_str  = ', '.join([w for w, _ in top_words[:5]])
    print(f'  Topik {topic_id:>2}: [{count:>3} pasal]  {words_str}')

# Simpan topic info
topic_info.to_csv('topic_modeling_result.csv', index=False, encoding='utf-8')
print('\n💾 topic_modeling_result.csv berhasil disimpan!')

2026-05-14 05:39:39,216 - BERTopic - Embedding - Transforming documents to embeddings.


🗺️  Mempersiapkan BERTopic...

⚙️  Menjalankan topic modeling pada 361 pasal (Bahasa Indonesia)...


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/12 [00:00<?, ?it/s]

2026-05-14 05:40:06,910 - BERTopic - Embedding - Completed ✓
2026-05-14 05:40:06,912 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2026-05-14 05:40:18,640 - BERTopic - Dimensionality - Completed ✓
2026-05-14 05:40:18,642 - BERTopic - Cluster - Start clustering the reduced embeddings
2026-05-14 05:40:18,671 - BERTopic - Cluster - Completed ✓
2026-05-14 05:40:18,672 - BERTopic - Representation - Extracting topics using c-TF-IDF for topic reduction.
2026-05-14 05:40:18,740 - BERTopic - Representation - Completed ✓
2026-05-14 05:40:18,741 - BERTopic - Topic reduction - Reducing number of topics
2026-05-14 05:40:18,757 - BERTopic - Representation - Fine-tuning topics using representation models.
2026-05-14 05:40:18,805 - BERTopic - Representation - Completed ✓
2026-05-14 05:40:18,807 - BERTopic - Topic reduction - Reduced number of topics from 50 to 33



✅ Topic modeling selesai!

📊 TOPIK HUKUM YANG DITEMUKAN:
  Topik  0: [ 67 pasal]  yang, korupsi, atau, tindak, dan
  Topik  1: [ 66 pasal]  dalam, sebagaimana, dimaksud, setiap, yang
  Topik  2: [ 23 pasal]  jelas, cukup, ayat, huruf, yang
  Topik  3: [ 17 pasal]  indonesia, republik, presiden, jakarta, tanggal
  Topik  4: [ 14 pasal]  paling, rupiah, juta, rp, lima
  Topik  5: [ 11 pasal]  korupsi, pemberantasan, tindak, komisi, pidana
  Topik  6: [ 10 pasal]  tentang, undangundang, 1981, acara, indonesia
  Topik  7: [  8 pasal]  melaksanakan, tugas, dalam, monitor, sebagaimana
  Topik  8: [  7 pasal]  wewenangnya, kekuasaan, tugas, lembaga, manapun
  Topik  9: [  6 pasal]  2019, 197, no, wwwperaturangoid, pimpinan
  Topik 10: [  6 pasal]  jawab, masyarakat, sesuai, penyelenggara, negara
  Topik 11: [  6 pasal]  memiliki, menjadi, selama, jabatan, sarjana
  Topik 12: [  6 pasal]  republikindonesia, cara, pelaporan, penentuan, presiden
  Topik 13: [  5 pasal]  pengawas, dewan, mengawa

## CELL 10 — Visualisasi Hasil

In [11]:
# ============================================================
# CELL 10 — VISUALISASI
# 1. Heatmap similarity UNCAC vs UU Indonesia
# 2. Pie chart status adopsi
# 3. Bar chart distribusi topik
# ============================================================

import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots

# ── VISUALISASI 1: Heatmap Similarity (UNCAC vs UU ID) ──
print('📊 Membuat visualisasi heatmap similarity...')

uncac_labels = [f"Art.{corpus_pasal[i]['pasal']}" for i in idx_uncac]
uu_labels    = [
    f"{corpus_pasal[i]['dokumen'].replace('_',' ')}\nPsl.{corpus_pasal[i]['pasal']}"
    for i in idx_uu_id
]

# Ambil subset untuk heatmap agar tidak terlalu besar
N_UNCAC = min(20, len(idx_uncac))
N_UU    = min(30, len(idx_uu_id))

fig_heat = go.Figure(data=go.Heatmap(
    z=sim_matrix[:N_UNCAC, :N_UU],
    x=uu_labels[:N_UU],
    y=uncac_labels[:N_UNCAC],
    colorscale='RdYlGn',
    zmin=0, zmax=1,
    colorbar=dict(title='Similarity Score')
))

fig_heat.update_layout(
    title='Heatmap Semantic Similarity: UNCAC vs UU Indonesia',
    xaxis_title='Pasal UU Indonesia',
    yaxis_title='UNCAC Article',
    height=600,
    font=dict(size=10)
)
fig_heat.show()


# ── VISUALISASI 2: Pie Chart Status Adopsi ──
print('📊 Membuat pie chart status adopsi...')

status_counts = df_gap['status'].value_counts()

fig_pie = px.pie(
    values=status_counts.values,
    names=status_counts.index,
    title='Status Adopsi Pasal UNCAC oleh UU Indonesia',
    color_discrete_map={
        'Diadopsi Penuh'      : '#2ecc71',
        'Diadopsi Sebagian'   : '#f39c12',
        'GAP - Belum Diadopsi': '#e74c3c'
    },
    hole=0.4
)
fig_pie.update_traces(textposition='inside', textinfo='percent+label')
fig_pie.show()


# ── VISUALISASI 3: Bar Chart Similarity per UNCAC Article ──
print('📊 Membuat bar chart similarity per artikel UNCAC...')

df_gap_plot = df_gap.copy().head(30)
df_gap_plot['color'] = df_gap_plot['status'].map({
    'Diadopsi Penuh'      : '#2ecc71',
    'Diadopsi Sebagian'   : '#f39c12',
    'GAP - Belum Diadopsi': '#e74c3c'
})

fig_bar = go.Figure(go.Bar(
    x=df_gap_plot['uncac_article'],
    y=df_gap_plot['similarity'],
    marker_color=df_gap_plot['color'],
    text=df_gap_plot['similarity'].round(3),
    textposition='outside'
))

fig_bar.add_hline(y=THRESHOLD_TINGGI,  line_dash='dash', line_color='green',
                  annotation_text='Diadopsi Penuh')
fig_bar.add_hline(y=THRESHOLD_SEDANG,  line_dash='dash', line_color='orange',
                  annotation_text='Diadopsi Sebagian')

fig_bar.update_layout(
    title='Similarity Score: Setiap Artikel UNCAC vs UU Indonesia Terbaik',
    xaxis_title='UNCAC Article',
    yaxis_title='Cosine Similarity Score',
    yaxis=dict(range=[0, 1.1]),
    height=500
)
fig_bar.show()


# ── VISUALISASI 4: Topic Modeling ──
print('📊 Membuat visualisasi topic modeling...')
try:
    fig_topics = topic_model.visualize_topics()
    fig_topics.show()
    fig_barchart = topic_model.visualize_barchart(top_n_topics=10)
    fig_barchart.show()
except Exception as e:
    print(f'   ⚠️  Visualisasi topic gagal: {e}')

print('\n✅ Semua visualisasi selesai!')

📊 Membuat visualisasi heatmap similarity...


📊 Membuat pie chart status adopsi...


📊 Membuat bar chart similarity per artikel UNCAC...


📊 Membuat visualisasi topic modeling...



✅ Semua visualisasi selesai!


## CELL 11 — Laporan Akhir & Export

In [12]:
# ============================================================
# CELL 11 — LAPORAN AKHIR & DOWNLOAD OUTPUT
# ============================================================

from google.colab import files

# Buat laporan ringkasan
laporan = {
    'judul'          : 'Analisis Normatif Hukum Korupsi Indonesia vs UNCAC',
    'tanggal'        : datetime.now().isoformat(),
    'ringkasan_corpus': {
        label: {
            'jumlah_pasal': len(pasal_per_dok.get(label, [])),
            'tahun'       : corpus_raw[label]['config']['tahun']
        }
        for label in corpus_raw
    },
    'gap_analysis': {
        'total_uncac_articles'   : len(df_gap),
        'diadopsi_penuh'         : n_penuh,
        'diadopsi_sebagian'      : n_sebagian,
        'gap_belum_diadopsi'     : n_gap,
        'threshold_penuh'        : THRESHOLD_TINGGI,
        'threshold_sebagian'     : THRESHOLD_SEDANG,
        'gap_terbesar'           : df_gap.sort_values('similarity').head(5)[[
            'uncac_article','similarity','status','padanan_terbaik'
        ]].to_dict('records')
    },
    'topic_modeling': {
        'jumlah_topik'   : len(topic_info[topic_info['Topic'] != -1]),
        'topik_teratas'  : [
            {
                'id'         : int(row['Topic']),
                'jumlah_pasal': int(row['Count']),
                'kata_kunci' : [w for w, _ in topic_model.get_topic(row['Topic'])[:5]]
            }
            for _, row in topic_info[topic_info['Topic'] != -1].head(5).iterrows()
        ]
    },
    'model_yang_digunakan': {
        'embedding'     : 'LaBSE (Language-agnostic BERT Sentence Embeddings)',
        'keyword'       : 'KeyBERT + paraphrase-multilingual-MiniLM-L12-v2',
        'topic_modeling': 'BERTopic + UMAP + HDBSCAN'
    }
}

with open('laporan_final.json', 'w', encoding='utf-8') as f:
    json.dump(laporan, f, ensure_ascii=False, indent=2)

print('📋 LAPORAN FINAL')
print('=' * 60)
print(f"  Dokumen dianalisis   : {len(corpus_raw)}")
print(f"  Total pasal          : {len(corpus_pasal)}")
print(f"  Artikel UNCAC        : {len(idx_uncac)}")
print(f"  Diadopsi Penuh       : {n_penuh} ({n_penuh/len(df_gap)*100:.1f}%)")
print(f"  Diadopsi Sebagian    : {n_sebagian} ({n_sebagian/len(df_gap)*100:.1f}%)")
print(f"  GAP (belum diadopsi) : {n_gap} ({n_gap/len(df_gap)*100:.1f}%)")
print(f"  Topik hukum ditemukan: {len(topic_info[topic_info['Topic'] != -1])}")
print('=' * 60)

# Download semua output
print('\n📥 Mendownload output files...')
for fname in ['corpus_structured.json', 'gap_analysis_result.csv',
              'topic_modeling_result.csv', 'laporan_final.json']:
    try:
        files.download(fname)
        print(f'   ✅ {fname}')
    except Exception as e:
        print(f'   ⚠️  {fname}: {e}')

print('\n🎉 Pipeline selesai! Project siap dipublikasikan ke GitHub.')

📋 LAPORAN FINAL
  Dokumen dianalisis   : 7
  Total pasal          : 501
  Artikel UNCAC        : 71
  Diadopsi Penuh       : 0 (0.0%)
  Diadopsi Sebagian    : 41 (57.7%)
  GAP (belum diadopsi) : 30 (42.3%)
  Topik hukum ditemukan: 32

📥 Mendownload output files...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

   ✅ corpus_structured.json


<IPython.core.display.Javascript object>

...

<IPython.core.display.Javascript object>

   ✅ topic_modeling_result.csv


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

   ✅ laporan_final.json

🎉 Pipeline selesai! Project siap dipublikasikan ke GitHub.
